# fall-guard-cv：XGBoost 跌倒偵測分類器訓練（Colab / T4）

輸入是本機已算好的 **視窗統計特徵**（54 維，來自 `scripts/prepare_train_export.py`），
不含任何影像或關鍵點座標——純數字，沒有隱私疑慮。

## 使用前請先上傳三個檔案（都在本機 repo 裡）
1. `data/export/xgb_windows.npz`（特徵 + 標籤，約 200KB）
2. `data/splits.json`（LOSO 切分定義）
3. `data/urfd_meta.csv`（受試者標註，選用，只用來印進度訊息）

## 訓練協定
- **LOSO（Leave-One-Subject-Out）**：5 折，每折用其餘受試者訓練、留一位受試者的影片當測試，
  折內防洩漏斷言（train/test 不共用任何 video_id）。
- 依 `docs/PLAN.md` D3：淺樹（max_depth 3–4）、數百棵，樣本少不追求深模型。
- 除了 5 折評估模型，另外用**全部資料**訓練一個「最終部署模型」，供 Phase 4 實際使用。
- 全部模型（5 折 + 1 最終）打包成一個 zip 下載，帶回本機用
  `uv run python scripts/evaluate.py --model xgb --protocol loso` 重現這裡印出的數字。

## 1. 安裝套件（版本需與本機一致，避免模型序列化不相容）

In [ ]:
!pip install -q "xgboost==3.2.0" "scikit-learn==1.9.0" "shap>=0.46,<1.0"
import xgboost, sklearn
print("xgboost", xgboost.__version__)
print("sklearn", sklearn.__version__)
assert xgboost.__version__ == "3.2.0", "xgboost 版本需與本機(3.2.0)一致，否則模型可能無法互相載入"


## 2. 上傳資料檔

In [ ]:
from google.colab import files
import io, json

print("請選擇 xgb_windows.npz、splits.json、urfd_meta.csv 三個檔案（可一次多選）")
uploaded = files.upload()
assert "xgb_windows.npz" in uploaded, "缺少 xgb_windows.npz"
assert "splits.json" in uploaded, "缺少 splits.json"


## 3. 載入資料 + LOSO 折防洩漏斷言（notebook 內防呆，不信任外部檔案）

In [ ]:
import numpy as np

data = np.load(io.BytesIO(uploaded["xgb_windows.npz"]), allow_pickle=True)
X, y, video_id = data["X"], data["y"], data["video_id"]
feature_names = [str(f) for f in data["feature_names"]]
print(f"視窗總數={len(y)}, 特徵維度={X.shape[1]}, 正例比例={y.mean():.1%}")

splits = json.loads(uploaded["splits.json"].decode("utf-8"))
loso_folds = splits["loso"]["folds"]
assert splits["loso"]["status"] == "ready", "splits.json 的 LOSO 尚未就緒"

# 防洩漏斷言：每折 train/test 的 video_id 不可重疊(不信任上傳檔案，本地重新驗證一次)
for fold in loso_folds:
    train_ids, test_ids = set(fold["train"]), set(fold["test"])
    assert not (train_ids & test_ids), f"fold {fold['subject']} train/test 有重疊，切分檔案可能損壞"
print(f"LOSO {len(loso_folds)} 折，防洩漏斷言全部通過")


## 4. LOSO 五折訓練 + 評估

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix
import xgboost as xgb

XGB_PARAMS = dict(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    min_child_weight=2,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    tree_method="hist",
    device="cuda",  # T4 GPU；本機 CPU 推論不受影響(推論不需要 GPU)
)

fold_models = {}
fold_metrics = []

for fold in loso_folds:
    subject = fold["subject"]
    train_mask = np.isin(video_id, fold["train"])
    test_mask = np.isin(video_id, fold["test"])
    X_train, y_train = X[train_mask], y[train_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    n_pos, n_neg = int(y_train.sum()), int((y_train == 0).sum())
    scale_pos_weight = n_neg / max(n_pos, 1)

    clf = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=scale_pos_weight, random_state=42)
    clf.fit(X_train, y_train, verbose=False)

    proba = clf.predict_proba(X_test)[:, 1]
    y_pred = (proba >= 0.5).astype(int)

    metrics = {
        "fold": subject,
        "n_train": len(y_train),
        "n_test": len(y_test),
        "precision": float(precision_score(y_test, y_pred, zero_division=0)),
        "recall": float(recall_score(y_test, y_pred, zero_division=0)),
        "f1": float(f1_score(y_test, y_pred, zero_division=0)),
        "pr_auc": float(average_precision_score(y_test, proba)) if len(set(y_test.tolist())) > 1 else None,
    }
    fold_metrics.append(metrics)
    fold_models[subject] = clf
    print(f"{subject}: n_test={metrics['n_test']} P={metrics['precision']:.3f} R={metrics['recall']:.3f} "
          f"F1={metrics['f1']:.3f} PR-AUC={metrics['pr_auc']}")

print()
print("=== LOSO 平均(mean ± std) ===")
for key in ["precision", "recall", "f1"]:
    vals = [m[key] for m in fold_metrics]
    print(f"{key}: {np.mean(vals):.3f} ± {np.std(vals):.3f}")


## 5. 最終部署模型（用全部 70 段影片訓練，供 Phase 4 使用）

這個模型不是用來評估的（沒有 held-out test），純粹是「用盡量多資料訓練出來、實際部署用」的版本，
跟規則式 baseline 的「評估用 N vs 部署用 N」分離設計（D11）是同一個精神。

In [ ]:
n_pos_all, n_neg_all = int(y.sum()), int((y == 0).sum())
final_clf = xgb.XGBClassifier(**XGB_PARAMS, scale_pos_weight=n_neg_all / max(n_pos_all, 1), random_state=42)
final_clf.fit(X, y, verbose=False)
print("最終部署模型訓練完成，樣本數:", len(y))


## 6. SHAP 特徵重要度（解釋模型學到了什麼）

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(final_clf)
shap_values = explainer.shap_values(X)

shap.summary_plot(shap_values, X, feature_names=feature_names, show=False, max_display=15)
plt.tight_layout()
plt.savefig("shap_summary.png", dpi=120)
plt.show()
print("已存 shap_summary.png（等下會一併打包下載）")


## 7. 打包下載（模型 + LOSO 結果 + SHAP 圖，帶回本機）

In [ ]:
import zipfile

for subject, clf in fold_models.items():
    clf.get_booster().save_model(f"xgb_fold_{subject}.json")
final_clf.get_booster().save_model("xgb_final.json")

with open("xgb_loso_results.json", "w", encoding="utf-8") as f:
    json.dump({"xgboost_version": xgboost.__version__, "params": XGB_PARAMS, "folds": fold_metrics}, f, ensure_ascii=False, indent=2)

zip_path = "xgb_models_bundle.zip"
with zipfile.ZipFile(zip_path, "w") as zf:
    for subject in fold_models:
        zf.write(f"xgb_fold_{subject}.json")
    zf.write("xgb_final.json")
    zf.write("xgb_loso_results.json")
    zf.write("shap_summary.png")

print(f"已打包 {zip_path}，即將觸發瀏覽器下載")
files.download(zip_path)


## 下一步（回到本機）

1. 解壓 `xgb_models_bundle.zip`，把 `xgb_fold_*.json` 與 `xgb_final.json` 放進本機 repo 的 `models/xgboost/`。
2. 跑 `uv run python scripts/evaluate.py --model xgb --protocol loso`，
   應該重現這裡印出的 P/R/F1（容許 ±0.01 誤差）。
3. 確認一致後，把 `models/xgboost/xgb_final.json` 上傳 Hugging Face
   （依 CLAUDE.md 權重不進 git 的規範；上傳前會先跟你確認）。